# 🧪 Prompt-Only Tool Calling — Can You Fake It Without the Native API?

### Dinesh AI Academy | Day 4 — Agents & MCP (Same Model, Two Techniques)

**The question this notebook answers:** in [3-Gemini_Tool_Calling_Raw_HTTP.ipynb](./3-Gemini_Tool_Calling_Raw_HTTP.ipynb)
we used Gemini's *native* `tools` field — a dedicated request field, a dedicated `functionCall`
response shape, schema-constrained decoding, all built into the API. **What if you don't use that
feature at all?** What if you just paste your tool schemas into the prompt as plain English and
JSON, and *ask* the model, in words, to reply in a specific format?

**Why we're using Gemini for both sides of this experiment, not OpenAI:** the obvious way to test
this is "send it to a different provider that has no native function-calling concept." But that
changes two variables at once — the *model* and the *technique*. To isolate the one variable that
actually matters here (native API feature vs. plain prompting), we run **the same model
(`gemini-3.5-flash-lite`) with the exact same tool set and exact same user questions**, once through
the native `tools` field (already captured in notebook 3) and once through **zero** API-level
tool-calling support — just text. Section 8 gives you the equivalent OpenAI request body, unexecuted,
to run yourself once your OpenAI credits are available; the prompt text itself is 100% portable
between providers because it's just words, not a provider-specific feature.

## 0. Three ways to get "which tool + which args" out of an LLM

| | A. Plain prompting (this notebook, Section 4) | B. Prompting + `responseSchema` (this notebook, Section 5+) | C. Native `tools` field (notebook 3) |
|---|---|---|---|
| Request field used | none — just text in `contents` | `generationConfig.responseMimeType` + `responseSchema` | `tools: [{functionDeclarations: [...]}]` |
| Guarantee the output is valid JSON at all | **None.** The model might wrap it in prose or a ```` ```json ```` fence | **Yes** — constrained decoding forces valid JSON matching your schema, token by token | **Yes** — same constrained-decoding machinery, applied to `functionCall` shape specifically |
| Guarantee the tool *name* is one you actually registered | None | Yes, if you add an `enum` to the schema | Yes — the model can only reference names from your `functionDeclarations` |
| Response shape | Whatever the model feels like | Exactly your schema, every time | `functionCall` / `text` parts, a shape the whole ecosystem (SDKs, `functionResponse` turns) agrees on |
| Who defined this shape | You, in English, hoping the model listens | You, in a formal JSON Schema | Google, as part of the Gemini API contract |

Notice B and C are both "constrained decoding" underneath — the difference is C additionally gives
you a *standardized, provider-supported* shape and conventions (like how to send the result back)
that the rest of the ecosystem (multi-turn `functionResponse`, automatic function calling, parallel
call arrays) is built to expect. B gets you valid JSON; it does not get you any of the *protocol*.

In [1]:
!pip -q install -U requests python-dotenv

## 1. Setup — same raw HTTP pattern as notebook 3, still zero SDK

In [1]:
import requests
import json
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")
if not GAISTUDIO_API_KEY:
    raise ValueError("GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file.")

MODEL = "gemini-3.5-flash-lite"  # SAME model as notebook 3 -- the only variable changing is technique
GENERATE_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def raw_call(payload: dict) -> dict:
    """Identical helper to notebook 3. Sends payload verbatim, returns Google's JSON verbatim.

    No retry logic on purpose -- see notebook 3's version of this function for why:
    you're running this interactively, so you are the retry loop, not this code.
    """
    response = requests.post(
        GENERATE_URL,
        params={"key": GAISTUDIO_API_KEY},
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    print(f"HTTP {response.status_code} {response.reason}")
    response.raise_for_status()
    return response.json()

print("Ready. Same model as notebook 3:", MODEL, "-- but the `tools` field will NEVER appear below.")

Ready. Same model as notebook 3: gemini-3.5-flash-lite -- but the `tools` field will NEVER appear below.


## 2. The same three tools, same real Python -- nothing about the tools changed

In [2]:
def get_weather(city: str) -> dict:
    WEATHER_DB = {
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    }
    data = WEATHER_DB.get(city.strip().lower(), {"temp_c": 20, "condition": "unknown"})
    return {"city": city, **data}

def celsius_to_fahrenheit(celsius: float) -> dict:
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, 1)}

def get_current_time(timezone: str) -> dict:
    from datetime import datetime
    from zoneinfo import ZoneInfo
    now = datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "current_time": now.strftime("%A, %d %B %Y, %I:%M:%S %p")}

TOOLBOX = {
    "get_weather": get_weather,
    "celsius_to_fahrenheit": celsius_to_fahrenheit,
    "get_current_time": get_current_time,
}
print("Toolbox ready:", list(TOOLBOX.keys()))

Toolbox ready: ['get_weather', 'celsius_to_fahrenheit', 'get_current_time']


## 3. Experiment A — plain prompting, zero format enforcement

We hand-write the *entire* tool-calling contract as English + embedded JSON, inside the prompt
text itself. No `tools` field. No `responseSchema`. Just an instruction and hope.

In [3]:
PROMPT_TEMPLATE = """You are a function-calling engine. You have access to the following tools.
You do not have access to anything else, and you cannot answer directly from your own knowledge
if a tool would answer the question more accurately.

TOOLS (JSON Schema):
[
  {{
    "name": "get_weather",
    "description": "Gets the current temperature (Celsius) and condition for a city.",
    "parameters": {{
      "type": "object",
      "properties": {{"city": {{"type": "string", "description": "City name, e.g. 'Tokyo'."}}}},
      "required": ["city"]
    }}
  }},
  {{
    "name": "celsius_to_fahrenheit",
    "description": "Converts a Celsius temperature to Fahrenheit.",
    "parameters": {{
      "type": "object",
      "properties": {{"celsius": {{"type": "number", "description": "Temperature in Celsius."}}}},
      "required": ["celsius"]
    }}
  }},
  {{
    "name": "get_current_time",
    "description": "Gets the current local date and time for an IANA timezone name.",
    "parameters": {{
      "type": "object",
      "properties": {{"timezone": {{"type": "string", "description": "IANA timezone, e.g. 'Asia/Tokyo'."}}}},
      "required": ["timezone"]
    }}
  }}
]

RESPONSE FORMAT -- you must output EXACTLY ONE JSON object, nothing else.
No markdown fences, no explanation before or after it, no extra keys.

If you need to call one or more tools, output exactly:
{{"type": "function_call", "calls": [{{"name": "<tool name>", "args": {{...}}}}, ...]}}

If you already have enough information to answer the user directly, output exactly:
{{"type": "text", "text": "<your final answer>"}}

USER QUESTION: {user_input}"""

payload_a = {
    "contents": [
        {"role": "user", "parts": [{"text": PROMPT_TEMPLATE.format(user_input="What's the weather in Mumbai right now?")}]}
    ]
    # <-- NOTICE: no "tools" key anywhere in this request. Google's servers have
    #     zero awareness that function calling is even the intent here.
}

raw_a = raw_call(payload_a)
print(json.dumps(raw_a, indent=2))

HTTP 200 OK
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "{\"type\": \"function_call\", \"calls\": [{\"name\": \"get_weather\", \"args\": {\"city\": \"Mumbai\"}}]}",
            "thoughtSignature": "El4KXAERTTIPLglOUFQm/hjX6s8lbOXy4XrGOqqmwVddbxqV3eZiWad4rrcHk49pcu0E22aVbK8GQbFwjldCpSW+hiNXLkCbC7vCT0Zz7cAUTKYzQD4shXB5oU9ECiFx"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 442,
    "candidatesTokenCount": 29,
    "totalTokenCount": 471,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 442
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "oyKsatr4Kur6juMPstjZsAc"
}


In [4]:
payload_a = {
    "contents": [
        {"role": "user", "parts": [{"text": PROMPT_TEMPLATE.format(user_input="What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?")}]}
    ]
    # <-- NOTICE: no "tools" key anywhere in this request. Google's servers have
    #     zero awareness that function calling is even the intent here.
}

raw_a = raw_call(payload_a)
print(json.dumps(raw_a, indent=2))

HTTP 200 OK
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "{\"type\": \"function_call\", \"calls\": [{\"name\": \"get_weather\", \"args\": {\"city\": \"Tokyo\"}}]}",
            "thoughtSignature": "El4KXAERTTIPeEyNHWaki+QDUdSCtud2X+V/ihDH1xIi97qSQUEu8uSM85ZIrj8y5pZ65Z3rZan4PfbREHxL6gJYcJPiyCzy78S7tuZvHIMVvgJhiuNWgu+PwCnMr9OH"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 450,
    "candidatesTokenCount": 29,
    "totalTokenCount": 479,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 450
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "AyOsasWdIcPp4-EPq8SNYA"
}


### Try to parse it -- honestly, without cleaning it up first

In [5]:
raw_text_a = raw_a["candidates"][0]["content"]["parts"][0]["text"]
print("Raw text the model returned:")
print(repr(raw_text_a))
print()

try:
    parsed_a = json.loads(raw_text_a)
    print("json.loads SUCCEEDED on the first try:", parsed_a)
except json.JSONDecodeError as e:
    print("json.loads FAILED:", e)
    print("This is the actual, honest failure mode of plain prompting: the model is not")
    print("structurally prevented from wrapping the JSON in a ```json fence, adding a")
    print("caveat sentence, or drifting from the schema. You would need a second, more")
    print("forgiving parser (regex-strip code fences, retry the request, etc.) in production.")

Raw text the model returned:
'{"type": "function_call", "calls": [{"name": "get_weather", "args": {"city": "Mumbai"}}]}'

json.loads SUCCEEDED on the first try: {'type': 'function_call', 'calls': [{'name': 'get_weather', 'args': {'city': 'Mumbai'}}]}


Whatever happened above is the **real, unedited result** — this notebook isn't going to pretend
plain prompting is perfectly reliable if it isn't. Even when it *does* produce valid JSON,
nothing forced it to; you got lucky that this particular prompt, on this particular call, landed
on compliant output. That's the whole problem with Technique A.

## 4. Experiment B — same prompt, now with `responseSchema` (constrained decoding, still no `tools`)

Gemini's `generationConfig.responseSchema` is a **separate feature from function calling**. It
forces the raw text output to be valid JSON matching a schema you provide, via the same kind of
constrained decoding notebook 3's `tools` field uses internally for `functionCall` args. Critically,
this proves constrained decoding and "is this a tool call" are two *independent* ideas that native
function calling happens to bundle together for you.

**One real limitation you hit immediately:** `functionDeclarations` lets each tool have its *own*
independent parameter schema. A single `responseSchema` for the whole response can't easily express
"one of three different, unrelated shapes depending on which tool got picked." We work around it by
flattening all possible arguments into one shared item shape -- a real, honest trade-off of this
technique, not swept under the rug.

In [6]:
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "type": {"type": "string", "enum": ["function_call", "text"]},
        "calls": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "enum": ["get_weather", "celsius_to_fahrenheit", "get_current_time"],
                    },
                    # Flattened args -- one field per possible parameter, across ALL tools.
                    # Native function calling never needs this hack: each functionDeclaration
                    # keeps its own separate `parameters` schema.
                    "city": {"type": "string"},
                    "celsius": {"type": "number"},
                    "timezone": {"type": "string"},
                },
                "required": ["name"],
            },
        },
        "text": {"type": "string"},
    },
    "required": ["type"],
}

payload_b = {
    "contents": [
        {"role": "user", "parts": [{"text": PROMPT_TEMPLATE.format(user_input="What's the weather in Mumbai right now?")}]}
    ],
    "generationConfig": {
        "responseMimeType": "application/json",
        "responseSchema": RESPONSE_SCHEMA,
    },
    # Still no "tools" key. Google's servers still have zero built-in concept of
    # "function calling" happening here -- only "produce JSON matching this shape."
}

raw_b = raw_call(payload_b)
print(json.dumps(raw_b, indent=2))

HTTP 200 OK
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "{\"type\": \"function_call\", \"calls\": [{\"name\": \"get_weather\", \"city\": \"Mumbai\"}]}",
            "thoughtSignature": "El4KXAERTTIPlmvdAE7rFm3rKTWKf1av36xl0Yyb3vanAQ8sjfkKN00qnHeogfBKW5jV1khWKGz4nGw89AO4lsjmpG6DxWfDo8D6IhWFP1C1Q/elWcH/wSa5swcdTZPO"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 442,
    "candidatesTokenCount": 26,
    "totalTokenCount": 468,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 442
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "KCCsaqf_OcP2juMPmJungQc"
}


In [7]:
raw_text_b = raw_b["candidates"][0]["content"]["parts"][0]["text"]
parsed_b = json.loads(raw_text_b)  # this time we expect it to just work, every time
print("Parsed cleanly:", parsed_b)

Parsed cleanly: {'type': 'function_call', 'calls': [{'name': 'get_weather', 'city': 'Mumbai'}]}


## 5. Wiring it into the SAME dispatcher as notebook 3

This is the point of the whole exercise: once you have a parsed dict, **your own dispatch code
does not care how the JSON was produced.** Same `TOOLBOX`, same `execute_tool` idea as notebook 3.

In [8]:
def execute_tool(name: str, args: dict):
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(f"Requested tool '{name}' is not in our TOOLBOX.")
    return func(**args)

def run_prompted_calls(parsed: dict):
    if parsed["type"] == "text":
        return parsed["text"]
    results = []
    for call in parsed["calls"]:
        name = call["name"]
        # Un-flatten: pick only the arg fields that this specific tool actually needs.
        arg_fields = {
            "get_weather": ["city"],
            "celsius_to_fahrenheit": ["celsius"],
            "get_current_time": ["timezone"],
        }[name]
        args = {k: call[k] for k in arg_fields if k in call}
        print(f"  -> executing {name}({args}) locally ...")
        results.append(execute_tool(name, args))
    return results

print(run_prompted_calls(parsed_b))

  -> executing get_weather({'city': 'Mumbai'}) locally ...
[{'city': 'Mumbai', 'temp_c': 31, 'condition': 'humid, partly cloudy'}]


## 6. Can prompt-only calling do *parallel* tool calls too?

Same independent-tools question as notebook 3's Section 10. Let's see if the `calls` array can
hold more than one entry when the model decides two unrelated tools are both needed.

In [9]:
payload_parallel = {
    "contents": [
        {"role": "user", "parts": [{
            "text": PROMPT_TEMPLATE.format(
                user_input="What's the weather in Paris, and separately, what time is it in "
                           "Asia/Tokyo? These two things are unrelated to each other."
            )
        }]}
    ],
    "generationConfig": {
        "responseMimeType": "application/json",
        "responseSchema": RESPONSE_SCHEMA,
    },
}

raw_parallel = raw_call(payload_parallel)
parsed_parallel = json.loads(raw_parallel["candidates"][0]["content"]["parts"][0]["text"])
print(json.dumps(parsed_parallel, indent=2))
print(f"\n{len(parsed_parallel.get('calls', []))} call(s) in the 'calls' array.")

HTTP 200 OK
{
  "type": "function_call",
  "calls": [
    {
      "name": "get_weather",
      "city": "Paris"
    },
    {
      "name": "get_current_time",
      "timezone": "Asia/Tokyo"
    }
  ]
}

2 call(s) in the 'calls' array.


## 7. Can you *force* tool-only output without `toolConfig`?

Notebook 3 forced a call with `toolConfig.functionCallingConfig.mode: "ANY"`. There's no such
field here -- but since `type` is governed entirely by our own `responseSchema`'s `enum`, we can get
the same effect by simply **removing `"text"` from the allowed enum values for `type`**. This is a
genuinely different mechanism (schema-level restriction vs. an API-level mode flag) that happens to
produce the same forcing behaviour.

In [10]:
FORCED_SCHEMA = json.loads(json.dumps(RESPONSE_SCHEMA))  # cheap deep copy
FORCED_SCHEMA["properties"]["type"]["enum"] = ["function_call"]  # "text" no longer a legal value

payload_forced = {
    "contents": [
        {"role": "user", "parts": [{
            "text": PROMPT_TEMPLATE.format(user_input="Tell me an interesting fact about octopuses.")
        }]}
    ],
    "generationConfig": {
        "responseMimeType": "application/json",
        "responseSchema": FORCED_SCHEMA,
    },
}

raw_forced = raw_call(payload_forced)
parsed_forced = json.loads(raw_forced["candidates"][0]["content"]["parts"][0]["text"])
print(json.dumps(parsed_forced, indent=2))

HTTP 200 OK
{
  "type": "function_call",
  "calls": []
}


### The real result is more interesting than a made-up one would have been

Look at what actually came back: `{"type": "function_call", "calls": []}` — an **empty array**,
not an invented `get_weather` call with a fake city. This is real, unedited output, and it exposes
a genuine gap between the two forcing mechanisms:

- Notebook 3's `toolConfig.functionCallingConfig.mode: "ANY"` is a **protocol-level guarantee**:
  the native function-calling contract says a `functionCall` part *will* be produced, so the model
  had to invent a city (it picked `"Seattle"`) — emptiness was never a legal output for that field.
- Our `responseSchema` only constrains the **shape** of the JSON (`type` must be the string
  `"function_call"`). It says nothing about `calls` needing at least one item, so the model found a
  perfectly schema-valid way to comply with the letter of our instruction while making no tool call
  at all: hand back `type: "function_call"` with zero calls in the array.

That's the real, practical cost of hand-rolling this yourself: you have to think of *and encode*
every constraint (here, `"minItems": 1` on the `calls` array) that native `tools` + `toolConfig`
gives you for free as part of the API's own contract. We didn't edit this result to make the point
neater — this is exactly what came back.

## 8. Side-by-side: native `tools` (notebook 3) vs. prompt-only + schema (this notebook)

These native-side JSON blocks are **quoted verbatim from notebook 3's real, executed output** —
no new API calls needed to make this comparison.

**Native (`tools` field, notebook 3, Section 5) — same question, same model:**
```json
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "get_weather",
              "args": {"city": "Mumbai"},
              "id": "call_41788"
            }
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "finishMessage": "Model generated function call(s)."
    }
  ]
}
```

**Prompt-only + `responseSchema` (this notebook, Section 4) — same question, same model:**
the `parts[0].text` field above contains a JSON *string* you had to parse yourself; there is no
`functionCall` key, no `finishMessage: "Model generated function call(s)."`, and the API itself has
no idea a "tool" was involved at all — it only knows it produced text matching your schema.

| | Native `tools` | Prompt-only + `responseSchema` |
|---|---|---|
| API knows a tool was selected | Yes — dedicated `functionCall` key, dedicated `finishMessage` | No — API sees only "valid JSON text", indistinguishable from any other structured output task |
| Valid-name guarantee | Built-in (model can only name a declared function) | Only if you remember to add an `enum` yourself |
| Per-tool independent parameter schema | Yes, free | No — you flatten everything into one shared shape yourself |
| Multi-turn `functionResponse` convention | Standardized, every SDK understands it | Doesn't exist — you'd have to invent and document your own "send the result back" turn shape |
| Automatic function-calling loops (SDK-side) | Exists, because the shape is standard | Cannot exist generically — every prompt's JSON shape is bespoke |
| Effort to reproduce this notebook's core outputs | ~10 lines: a `tools` field | ~50 lines: a hand-written contract, a hand-written parser, a hand-written schema, a hand-written unflattening step |

**The honest conclusion:** prompting alone, even reinforced with `responseSchema`, gets remarkably
close for a single-provider prototype. What you permanently give up is the *protocol* — the
standardized shape that every framework, every SDK's automatic-calling loop, and every other
engineer already knows how to consume without reading your custom instructions.

## 9. The exact same technique against OpenAI — ready to run once credits are available

`OPENAI_API_KEY` in this project's `.env` currently has no remaining credits (`insufficient_quota`
was the real error when this was tested), so this cell is provided **unexecuted** — the prompt text
is copy-pasted verbatim from Section 3/4 above, because that text is plain English + JSON and has
no dependency on Gemini at all. Only the HTTP envelope changes (OpenAI uses `messages`, not
`contents`; `response_format`, not `generationConfig.responseSchema`).

In [11]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
OPENAI_URL = "https://api.openai.com/v1/chat/completions"

openai_payload = {
    "model": "gpt-4o-mini",
    "messages": [
        {"role": "user", "content": PROMPT_TEMPLATE.format(user_input="What's the weather in Mumbai right now?")}
    ],
    "response_format": {"type": "json_object"},  # OpenAI's own "guarantee valid JSON" switch --
                                                      # note it does NOT accept a full JSON Schema
                                                      # here the way Gemini's responseSchema does;
                                                      # it only guarantees *parseable* JSON, not
                                                      # conformance to your exact shape.
}

print("This is the exact request that will run once OPENAI_API_KEY has credits:")
print(json.dumps(openai_payload, indent=2))

response = requests.post(
    OPENAI_URL,
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
    json=openai_payload,
    timeout=60,
)
print(f"\nHTTP {response.status_code} {response.reason}")
print(response.text[:1500])
if response.ok:
    openai_text = response.json()["choices"][0]["message"]["content"]
    print("\nParsed:", json.loads(openai_text))

This is the exact request that will run once OPENAI_API_KEY has credits:
{
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "user",
      "content": "You are a function-calling engine. You have access to the following tools.\nYou do not have access to anything else, and you cannot answer directly from your own knowledge\nif a tool would answer the question more accurately.\n\nTOOLS (JSON Schema):\n[\n  {\n    \"name\": \"get_weather\",\n    \"description\": \"Gets the current temperature (Celsius) and condition for a city.\",\n    \"parameters\": {\n      \"type\": \"object\",\n      \"properties\": {\"city\": {\"type\": \"string\", \"description\": \"City name, e.g. 'Tokyo'.\"}},\n      \"required\": [\"city\"]\n    }\n  },\n  {\n    \"name\": \"celsius_to_fahrenheit\",\n    \"description\": \"Converts a Celsius temperature to Fahrenheit.\",\n    \"parameters\": {\n      \"type\": \"object\",\n      \"properties\": {\"celsius\": {\"type\": \"number\", \"description\": \"T


HTTP 429 Too Many Requests
{
    "error": {
        "message": "You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.",
        "type": "insufficient_quota",
        "param": null,
        "code": "credit_balance_exhausted"
    }
}



## 🎓 Takeaway

1. **Function calling is not one indivisible feature.** It's (a) constrained decoding for valid
   JSON, plus (b) a standardized shape and multi-turn protocol on top. Native `tools` bundles both;
   you can get (a) alone via `responseSchema`/`response_format`, and you can attempt neither and get
   (nothing guaranteed) via plain prompting.
2. **Plain prompting (Section 3) is genuinely unreliable** — you saw the real parse result, not a
   hypothetical one. It's fine for a demo, risky for production.
3. **`responseSchema` (Section 4 onward) closes the reliability gap** but exposes real limitations
   native tool calling doesn't have: one shared schema instead of per-tool schemas, no built-in
   `functionResponse` convention, no automatic-calling loop support in any SDK.
4. **This entire experiment is provider-agnostic by design** — Section 9 is the same prompt text,
   ready to run against OpenAI the moment credits exist, with zero changes to `PROMPT_TEMPLATE` or
   `RESPONSE_SCHEMA`'s *ideas* (only the HTTP envelope around them differs per provider).
5. If you ever have to integrate tool-calling behaviour with a model or endpoint that has no native
   `tools`/`functions` support at all, this notebook is the actual technique you'd reach for --
   now you've seen exactly what it costs you compared to the native path.

## Official references

- Gemini API — Structured output (`responseSchema`): https://ai.google.dev/gemini-api/docs/structured-output
- Gemini API — Function calling guide: https://ai.google.dev/gemini-api/docs/function-calling
- OpenAI — Structured outputs / `response_format`: https://platform.openai.com/docs/guides/structured-outputs
- OpenAI — Function calling: https://platform.openai.com/docs/guides/function-calling

See also: [3-Gemini_Tool_Calling_Raw_HTTP.ipynb](./3-Gemini_Tool_Calling_Raw_HTTP.ipynb) for the
native-`tools` baseline every comparison in this notebook is measured against.